## Import Libraries

In [11]:
import os
from google.colab import drive
drive.mount('/content/drive')
import numpy as np, random
from tqdm.auto import tqdm
import re
import ast
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from collections import Counter
import pickle
import openpyxl
from openpyxl import load_workbook
import shutil
import glob


!pip install transformers datasets scikit-learn tqdm joblib --quiet
import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import BertTokenizerFast, BertForSequenceClassification
from transformers import logging as transformers_logging
from transformers import get_linear_schedule_with_warmup


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Preprocess

In [3]:
# Paths to data folders
annotations_dir = '/content/drive/MyDrive/Colab Notebooks/Project/English_consolidation/'
texts_dir = '/content/drive/MyDrive/Colab Notebooks/Project/English_sanitized_policies/'


# List .csv or .tsv files in the annotations_dir
csv_files = [f for f in os.listdir(annotations_dir) if f.endswith(('.csv', '.tsv'))]

sample_csv = pd.read_csv(os.path.join(annotations_dir, csv_files[0]), sep='\t', engine='python')
print(sample_csv.columns.tolist())


['policy id,segment_id,category name,attribute name,value name,policy_type,privacy_policy_link,MAPP_59']


In [4]:
# Create an empty list to store all DataFrames
all_dfs = []

# Loop through each CSV and match its TXT
for csv_file in tqdm(csv_files, desc="Processing files"):
    csv_path = os.path.join(annotations_dir, csv_file)
    txt_file = csv_file.replace('.csv', '.txt')
    txt_path = os.path.join(texts_dir, txt_file)

    # Check if the matching text file exists
    if not os.path.exists(txt_path):
        print(f"Text file missing for: {csv_file}")
        continue

    # Load CSV
    df = pd.read_csv(csv_path, sep=',', engine='python')
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_') # normalize column headers

    # Read and segment the text file (split by double newlines)
    with open(txt_path, 'r', encoding='utf-8') as f:
        text = f.read()
        segments = [seg.strip() for seg in text.replace('\r\n', '\n').replace('\r', '\n').split('\n\n') if seg.strip()]

    # Map segment text to DataFrame
    df['segment_text'] = df['segment_id'].apply(
        lambda x: segments[x - 1] if 1 <= x <= len(segments) else ""
    )

    # Add filename as an identifier (optional)
    df['source_file'] = csv_file

    # Append processed DataFrame to list
    all_dfs.append(df)

# Combine all into one DataFrame
final_df = pd.concat(all_dfs, ignore_index=True)

# Done
print("n/All files processed. Final DataFrame shape:", final_df.shape)


Processing files:   0%|          | 0/64 [00:00<?, ?it/s]

n/All files processed. Final DataFrame shape: (3976, 10)


In [5]:
# Extract APP_ID from source_file in final_df
final_df['APP_ID'] = final_df['source_file'].str.extract(r'_(.*)\.csv')

final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3976 entries, 0 to 3975
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   policy_id            3976 non-null   int64 
 1   segment_id           3976 non-null   int64 
 2   category_name        3976 non-null   object
 3   attribute_name       3976 non-null   object
 4   value_name           3976 non-null   object
 5   policy_type          3976 non-null   object
 6   privacy_policy_link  3976 non-null   object
 7   mapp_59              3976 non-null   bool  
 8   segment_text         3976 non-null   object
 9   source_file          3976 non-null   object
 10  APP_ID               3976 non-null   object
dtypes: bool(1), int64(2), object(8)
memory usage: 314.6+ KB


In [6]:
# Save merged data
final_df.to_csv("mapp_preprocessed_dataset2.csv", index=False)

# Check data
print("Unique policies:", final_df['policy_id'].nunique())

missing_segments = final_df[final_df['segment_text'].isnull()]
print("Total missing segments:", len(missing_segments))

duplicates =  final_df[final_df.duplicated(subset=['policy_id', 'segment_id', 'category_name', 'attribute_name', 'segment_text'], keep=False)]
print("Total duplicates:", len(duplicates))

# Check total segments
text_files = [f for f in os.listdir(texts_dir) if f.endswith('.txt')]

total_segments = 0

for txt_file in sorted(text_files):
    with open(os.path.join(texts_dir, txt_file), 'r', encoding='utf-8') as f:
        text = f.read()
        segments = [seg.strip() for seg in re.split(r'\n\s*\n', text) if seg.strip()]
        num_segments = len(segments)
        total_segments += num_segments

print(f"Total segments across all TXT files: {total_segments}")


Unique policies: 64
Total missing segments: 0
Total duplicates: 0
Total segments across all TXT files: 3976


In [7]:
sam = final_df.copy()

# list of all unique value strings
sam['value_name'] = sam['value_name'].apply(lambda x: ast.literal_eval(x))
all_labels = set()
for labels in sam['value_name']:
    all_labels.update(labels)
all_labels = sorted(list(all_labels))
all_labels

['Anonymization (opt)_Aggregated or anonymized',
 'Anonymization (opt)_Identifiable',
 'Choice Scope (opt)_Use',
 'Choice Type (opt)_Browser/device privacy controls',
 "Choice Type (opt)_Don't use service/feature",
 'Choice Type (opt)_First-party privacy controls',
 'Choice Type (opt)_Opt-in',
 'Choice Type (opt)_Opt-out link',
 'Choice Type (opt)_Opt-out via contacting company',
 'Choice Type (opt)_Other',
 'Choice Type (opt)_Third-party privacy controls',
 'Collection Mode (opt)_Explicit',
 'Collection Mode (opt)_Implicit',
 'Collection Process_Collected on first-party website/app',
 'Collection Process_Directly received from third party',
 'Collection Process_Entered by user on first-party website/app',
 'Collection Process_Entered by user on third-party website/app',
 'Collection Process_Other',
 'Collection Process_Shared by first party with a third party',
 'Collection Process_Tracked on first-party website/app by third party',
 'Collection Process_Tracked user on third-party web

## Train and Validate

In [ ]:
# Config
MAX_LEN = 512
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATES = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5]
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Save directory
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/win")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
excel_path = "/content/drive/MyDrive/Colab Notebooks/Project/win/training_results_bert.xlsx"
val_save_path = MODEL_DIR / f"val_set_{safe_label}.pkl"


# for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Filter and prepare
sam = sam[sam['value_name'].apply(lambda x: len(x) > 0)]
unique_labels = sorted(set(label for sublist in sam['value_name'] for label in sublist))

custom_labels = [
    'Information Type_Contact information',
    'Purpose_Essential service or feature',
    'Information Type_Personal identifier',

    #"Information Type_IP address and device IDs",
    #"Information Type_Location",
    #"Information Type_Health, genetic, or biometric data",

    #"Information Type_Computer information",
    #"Information Type_User online activities",
    #"Information Type_Generic personal information",

    #"Collection Process_Collected on first-party website/app",
    #"Collection Process_Shared by first party with a third party",
    #"Purpose_Advertising or marketing",

    #"Purpose_Analytics or research",
    #"Purpose_Service operation and security",
    #"Purpose_Legal requirement"

]

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

# Evaluation
def evaluate_model(model, dataloader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits.squeeze())
            preds.extend(probs.cpu().numpy())

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall

# Optimize with higher lr
def get_optimizer(model, base_lr, classifier_lr=1e-4, weight_decay=0.01):
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': 0.0
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': 0.0
        },
    ]
    return AdamW(optimizer_grouped_parameters)

# Training
def train_model(lr, train_loader, pos_weight_val):
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([pos_weight_val]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(train_loader, desc=f"LR {lr:.0e} | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Final retrain
def retrain_best_model(best_lr, best_pos_weight, full_train_loader):
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=best_lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(full_train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([best_pos_weight]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(full_train_loader, desc=f"Retrain | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Training Loop
all_results = []

for label_to_train in custom_labels:
    safe_label = label_to_train.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_bert_model_{safe_label}.pt"

    if model_path.exists():
        print(f" Skipping {label_to_train} — model already exists.")
        continue

    print(f"\nTraining for: {label_to_train}")

    train_df = sam[sam['policy_type'] == 'TRAIN'].copy()
    train_df['binary_label'] = train_df['value_name'].apply(lambda x: 1 if label_to_train in x else 0)

    positives = train_df[train_df['binary_label'] == 1]
    negatives = train_df[train_df['binary_label'] == 0]
    if len(positives) < 5:
        print(f" Too few positives ({len(positives)}). Skipping.")
        continue

    # pos_weight on original training data
    label_counts = Counter(train_df['binary_label'].tolist())
    pos_weight = label_counts[0] / label_counts[1]
    print(f" Using pos_weight: {pos_weight:.2f}")

    # Train/val split
    texts = train_df['segment_text'].tolist()
    labels = train_df['binary_label'].tolist()
    texts_train, texts_val, y_train, y_val = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=SEED
    )
    with open(val_save_path, "wb") as f:
        pickle.dump({"texts": texts_val, "labels": y_val}, f)

    # Oversample positives in training
    train_data = pd.DataFrame({'segment_text': texts_train, 'binary_label': y_train})
    positives_train = train_data[train_data['binary_label'] == 1]
    negatives_train = train_data[train_data['binary_label'] == 0]

    max_oversample_size = min(len(negatives_train), len(positives_train) * 4)
    oversampled_positives = positives_train.sample(max_oversample_size, replace=True, random_state=SEED)
    train_df_balanced = pd.concat([negatives_train, oversampled_positives], ignore_index=True).sample(frac=1, random_state=SEED)

    # Build loaders
    train_dataset = TextDataset(train_df_balanced['segment_text'].tolist(), train_df_balanced['binary_label'].tolist())
    val_dataset = TextDataset(texts_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # Find best model
    best_f1 = 0
    best_lr = None
    for lr in LEARNING_RATES:
        model = train_model(lr, train_loader, pos_weight)
        acc, f1, precision, recall = evaluate_model(model, val_loader)

        if f1 > best_f1:
            best_f1 = f1
            best_lr = lr
            torch.save(model.state_dict(), model_path)
            print(f"Best model updated and saved at LR {lr} for label {label_to_train}")

        all_results.append({
            'label': label_to_train,
            'learning_rate': lr,
            'pos_weight': round(pos_weight, 2),
            'f1': round(f1, 4),
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'best_model': 'best' if f1 == best_f1 else ''
        })

        # Save results to Excel immediately after every LR
        all_models_df = pd.DataFrame(all_results)

        # Append or create "All Models" sheet
        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "All Models" in book.sheetnames:
                existing_df = pd.read_excel(excel_path, sheet_name="All Models")
                combined_df = pd.concat([existing_df, all_models_df], ignore_index=True)
                combined_df.drop_duplicates(subset=['label', 'learning_rate'], inplace=True)
            else:
                combined_df = all_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_df.to_excel(writer, sheet_name="All Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                all_models_df.to_excel(writer, sheet_name="All Models", index=False)

        # Append or create "Best Models" sheet
        best_models_df = all_models_df.sort_values('f1', ascending=False).drop_duplicates('label')

        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "Best Models" in book.sheetnames:
                existing_best_df = pd.read_excel(excel_path, sheet_name="Best Models")
                combined_best_df = pd.concat([existing_best_df, best_models_df], ignore_index=True)
                combined_best_df.drop_duplicates(subset=['label'], inplace=True)
            else:
                combined_best_df = best_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_best_df.to_excel(writer, sheet_name="Best Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                best_models_df.to_excel(writer, sheet_name="Best Models", index=False)



    # Final retrain on full dataset
    full_dataset = TextDataset(train_df['segment_text'].tolist(), train_df['binary_label'].tolist())
    full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"\n Retraining best model for {label_to_train} on full data...")
    best_model = retrain_best_model(best_lr, pos_weight, full_loader)

    torch.save(best_model.state_dict(), model_path)
    print(f" Final model saved to {model_path}")


In [ ]:
MAX_LEN = 512
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATES = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5]
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Save directory
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/win")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
excel_path = "/content/drive/MyDrive/Colab Notebooks/Project/win/training_results_bert.xlsx"
val_save_path = MODEL_DIR / f"val_set_{safe_label}.pkl"


# for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Filter and prepare
sam = sam[sam['value_name'].apply(lambda x: len(x) > 0)]
unique_labels = sorted(set(label for sublist in sam['value_name'] for label in sublist))

custom_labels = [
    #'Information Type_Contact information',
    #'Purpose_Essential service or feature',
    #'Information Type_Personal identifier',

    "Information Type_IP address and device IDs",
    "Information Type_Location",
    "Information Type_Health, genetic, or biometric data",

    "Information Type_Computer information",
    #"Information Type_User online activities",
    #"Information Type_Generic personal information",

    #"Collection Process_Collected on first-party website/app",
    #"Collection Process_Shared by first party with a third party",
    #"Purpose_Advertising or marketing",

    #"Purpose_Analytics or research",
    #"Purpose_Service operation and security",
    #"Purpose_Legal requirement"

]

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

# Evaluation
def evaluate_model(model, dataloader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits.squeeze())
            preds.extend(probs.cpu().numpy())

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall

# Optimize with a higher lr
def get_optimizer(model, base_lr, classifier_lr=1e-4, weight_decay=0.01):
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': 0.0
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': 0.0
        },
    ]
    return AdamW(optimizer_grouped_parameters)

# Training
def train_model(lr, train_loader, pos_weight_val):
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([pos_weight_val]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(train_loader, desc=f"LR {lr:.0e} | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Final retrain
def retrain_best_model(best_lr, best_pos_weight, full_train_loader):
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=best_lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(full_train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([best_pos_weight]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(full_train_loader, desc=f"Retrain | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Training Loop
all_results = []

for label_to_train in custom_labels:
    safe_label = label_to_train.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_bert_model_{safe_label}.pt"

    if model_path.exists():
        print(f" Skipping {label_to_train} — model already exists.")
        continue

    print(f"\nTraining for: {label_to_train}")

    train_df = sam[sam['policy_type'] == 'TRAIN'].copy()
    train_df['binary_label'] = train_df['value_name'].apply(lambda x: 1 if label_to_train in x else 0)

    positives = train_df[train_df['binary_label'] == 1]
    negatives = train_df[train_df['binary_label'] == 0]
    if len(positives) < 5:
        print(f" Too few positives ({len(positives)}). Skipping.")
        continue

    # pos_weight on original training data
    label_counts = Counter(train_df['binary_label'].tolist())
    pos_weight = label_counts[0] / label_counts[1]
    print(f" Using pos_weight: {pos_weight:.2f}")

    # Train/val split
    texts = train_df['segment_text'].tolist()
    labels = train_df['binary_label'].tolist()
    texts_train, texts_val, y_train, y_val = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=SEED
    )
    with open(val_save_path, "wb") as f:
        pickle.dump({"texts": texts_val, "labels": y_val}, f)

    # Oversample positives in training
    train_data = pd.DataFrame({'segment_text': texts_train, 'binary_label': y_train})
    positives_train = train_data[train_data['binary_label'] == 1]
    negatives_train = train_data[train_data['binary_label'] == 0]

    max_oversample_size = min(len(negatives_train), len(positives_train) * 4)
    oversampled_positives = positives_train.sample(max_oversample_size, replace=True, random_state=SEED)
    train_df_balanced = pd.concat([negatives_train, oversampled_positives], ignore_index=True).sample(frac=1, random_state=SEED)

    # Build loaders
    train_dataset = TextDataset(train_df_balanced['segment_text'].tolist(), train_df_balanced['binary_label'].tolist())
    val_dataset = TextDataset(texts_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # Find best model
    best_f1 = 0
    best_lr = None
    for lr in LEARNING_RATES:
        model = train_model(lr, train_loader, pos_weight)
        acc, f1, precision, recall = evaluate_model(model, val_loader)

        if f1 > best_f1:
            best_f1 = f1
            best_lr = lr
            torch.save(model.state_dict(), model_path)
            print(f"Best model updated and saved at LR {lr} for label {label_to_train}")

        all_results.append({
            'label': label_to_train,
            'learning_rate': lr,
            'pos_weight': round(pos_weight, 2),
            'f1': round(f1, 4),
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'best_model': 'best' if f1 == best_f1 else ''
        })

        # Save results to Excel immediately after every LR
        all_models_df = pd.DataFrame(all_results)

        # Append or create "All Models" sheet
        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "All Models" in book.sheetnames:
                existing_df = pd.read_excel(excel_path, sheet_name="All Models")
                combined_df = pd.concat([existing_df, all_models_df], ignore_index=True)
                combined_df.drop_duplicates(subset=['label', 'learning_rate'], inplace=True)
            else:
                combined_df = all_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_df.to_excel(writer, sheet_name="All Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                all_models_df.to_excel(writer, sheet_name="All Models", index=False)

        # Append or create "Best Models" sheet
        best_models_df = all_models_df.sort_values('f1', ascending=False).drop_duplicates('label')

        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "Best Models" in book.sheetnames:
                existing_best_df = pd.read_excel(excel_path, sheet_name="Best Models")
                combined_best_df = pd.concat([existing_best_df, best_models_df], ignore_index=True)
                combined_best_df.drop_duplicates(subset=['label'], inplace=True)
            else:
                combined_best_df = best_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_best_df.to_excel(writer, sheet_name="Best Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                best_models_df.to_excel(writer, sheet_name="Best Models", index=False)



    # Final retrain on full dataset
    full_dataset = TextDataset(train_df['segment_text'].tolist(), train_df['binary_label'].tolist())
    full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"\n Retraining best model for {label_to_train} on full data...")
    best_model = retrain_best_model(best_lr, pos_weight, full_loader)

    torch.save(best_model.state_dict(), model_path)
    print(f" Final model saved to {model_path}")


In [ ]:
MAX_LEN = 512
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATES = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5]
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Save directory
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/win")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
excel_path = "/content/drive/MyDrive/Colab Notebooks/Project/win/training_results_bert.xlsx"
val_save_path = MODEL_DIR / f"val_set_{safe_label}.pkl"


# for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Filter and prepare
sam = sam[sam['value_name'].apply(lambda x: len(x) > 0)]
unique_labels = sorted(set(label for sublist in sam['value_name'] for label in sublist))

custom_labels = [
    #'Information Type_Contact information',
    #'Purpose_Essential service or feature',
    #'Information Type_Personal identifier',

    #"Information Type_IP address and device IDs",
    #"Information Type_Location",
    #"Information Type_Health, genetic, or biometric data",

    #"Information Type_Computer information",
    "Information Type_User online activities",
    "Information Type_Generic personal information",

    "Collection Process_Collected on first-party website/app",
    "Collection Process_Shared by first party with a third party",
    #"Purpose_Advertising or marketing",

    #"Purpose_Analytics or research",
    #"Purpose_Service operation and security",
    #"Purpose_Legal requirement"

]



# Dataset Class
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

# Evaluation
def evaluate_model(model, dataloader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits.squeeze())
            preds.extend(probs.cpu().numpy())

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall

# Optimize
def get_optimizer(model, base_lr, classifier_lr=1e-4, weight_decay=0.01):
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': 0.0
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': 0.0
        },
    ]
    return AdamW(optimizer_grouped_parameters)

# Training
def train_model(lr, train_loader, pos_weight_val):
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([pos_weight_val]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(train_loader, desc=f"LR {lr:.0e} | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Final retrain
def retrain_best_model(best_lr, best_pos_weight, full_train_loader):
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=best_lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(full_train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([best_pos_weight]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(full_train_loader, desc=f"Retrain | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Training Loop
all_results = []

for label_to_train in custom_labels:
    safe_label = label_to_train.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_bert_model_{safe_label}.pt"

    if model_path.exists():
        print(f" Skipping {label_to_train} — model already exists.")
        continue

    print(f"\nTraining for: {label_to_train}")

    train_df = sam[sam['policy_type'] == 'TRAIN'].copy()
    train_df['binary_label'] = train_df['value_name'].apply(lambda x: 1 if label_to_train in x else 0)

    positives = train_df[train_df['binary_label'] == 1]
    negatives = train_df[train_df['binary_label'] == 0]
    if len(positives) < 5:
        print(f" Too few positives ({len(positives)}). Skipping.")
        continue

    # pos_weight on original training data
    label_counts = Counter(train_df['binary_label'].tolist())
    pos_weight = label_counts[0] / label_counts[1]
    print(f" Using pos_weight: {pos_weight:.2f}")

    # Train/val split
    texts = train_df['segment_text'].tolist()
    labels = train_df['binary_label'].tolist()
    texts_train, texts_val, y_train, y_val = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=SEED
    )
    with open(val_save_path, "wb") as f:
        pickle.dump({"texts": texts_val, "labels": y_val}, f)

    # Oversample positives in training
    train_data = pd.DataFrame({'segment_text': texts_train, 'binary_label': y_train})
    positives_train = train_data[train_data['binary_label'] == 1]
    negatives_train = train_data[train_data['binary_label'] == 0]

    max_oversample_size = min(len(negatives_train), len(positives_train) * 4)
    oversampled_positives = positives_train.sample(max_oversample_size, replace=True, random_state=SEED)
    train_df_balanced = pd.concat([negatives_train, oversampled_positives], ignore_index=True).sample(frac=1, random_state=SEED)

    # Build loaders
    train_dataset = TextDataset(train_df_balanced['segment_text'].tolist(), train_df_balanced['binary_label'].tolist())
    val_dataset = TextDataset(texts_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # Find best model
    best_f1 = 0
    best_lr = None
    for lr in LEARNING_RATES:
        model = train_model(lr, train_loader, pos_weight)
        acc, f1, precision, recall = evaluate_model(model, val_loader)

        if f1 > best_f1:
            best_f1 = f1
            best_lr = lr
            torch.save(model.state_dict(), model_path)
            print(f"Best model updated and saved at LR {lr} for label {label_to_train}")

        all_results.append({
            'label': label_to_train,
            'learning_rate': lr,
            'pos_weight': round(pos_weight, 2),
            'f1': round(f1, 4),
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'best_model': 'best' if f1 == best_f1 else ''
        })

        # Save results to Excel immediately after every LR
        all_models_df = pd.DataFrame(all_results)

        # Append or create "All Models" sheet
        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "All Models" in book.sheetnames:
                existing_df = pd.read_excel(excel_path, sheet_name="All Models")
                combined_df = pd.concat([existing_df, all_models_df], ignore_index=True)
                combined_df.drop_duplicates(subset=['label', 'learning_rate'], inplace=True)
            else:
                combined_df = all_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_df.to_excel(writer, sheet_name="All Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                all_models_df.to_excel(writer, sheet_name="All Models", index=False)

        # Append or create "Best Models" sheet
        best_models_df = all_models_df.sort_values('f1', ascending=False).drop_duplicates('label')

        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "Best Models" in book.sheetnames:
                existing_best_df = pd.read_excel(excel_path, sheet_name="Best Models")
                combined_best_df = pd.concat([existing_best_df, best_models_df], ignore_index=True)
                combined_best_df.drop_duplicates(subset=['label'], inplace=True)
            else:
                combined_best_df = best_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_best_df.to_excel(writer, sheet_name="Best Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                best_models_df.to_excel(writer, sheet_name="Best Models", index=False)



    # Final retrain on full dataset
    full_dataset = TextDataset(train_df['segment_text'].tolist(), train_df['binary_label'].tolist())
    full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"\n Retraining best model for {label_to_train} on full data...")
    best_model = retrain_best_model(best_lr, pos_weight, full_loader)

    torch.save(best_model.state_dict(), model_path)
    print(f" Final model saved to {model_path}")


In [ ]:
# Config
MAX_LEN = 512
BATCH_SIZE = 32
EPOCHS = 3
LEARNING_RATES = [5e-6, 1e-5, 2e-5, 3e-5, 5e-5]
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Save directory
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/win")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
excel_path = "/content/drive/MyDrive/Colab Notebooks/Project/win/training_results_bert.xlsx"
val_save_path = MODEL_DIR / f"val_set_{safe_label}.pkl"

# For reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Tokenizer
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

# Filter and prepare
sam = sam[sam['value_name'].apply(lambda x: len(x) > 0)]
unique_labels = sorted(set(label for sublist in sam['value_name'] for label in sublist))

custom_labels = [
    #'Information Type_Contact information',
    #'Purpose_Essential service or feature',
    #'Information Type_Personal identifier',

    #"Information Type_IP address and device IDs",
    #"Information Type_Location",
    #"Information Type_Health, genetic, or biometric data",

    #"Information Type_Computer information",
    #"Information Type_User online activities",
    #"Information Type_Generic personal information",

    #"Collection Process_Collected on first-party website/app",
    #"Collection Process_Shared by first party with a third party",
    "Purpose_Advertising or marketing",

    "Purpose_Analytics or research",
    "Purpose_Service operation and security",
    "Purpose_Legal requirement"

]


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

# Evaluation
def evaluate_model(model, dataloader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits.squeeze())
            preds.extend(probs.cpu().numpy())

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall

# Optimizer helper
def get_optimizer(model, base_lr, classifier_lr=1e-4, weight_decay=0.01):
    no_decay = ['bias', 'LayerNorm.weight']
    optimizer_grouped_parameters = [
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' not in n],
            'lr': base_lr,
            'weight_decay': 0.0
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if not any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': weight_decay
        },
        {
            'params': [p for n, p in model.named_parameters()
                       if any(nd in n for nd in no_decay) and 'classifier' in n],
            'lr': classifier_lr,
            'weight_decay': 0.0
        },
    ]
    return AdamW(optimizer_grouped_parameters)

# Training
def train_model(lr, train_loader, pos_weight_val):
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([pos_weight_val]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(train_loader, desc=f"LR {lr:.0e} | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

# Final retrain
def retrain_best_model(best_lr, best_pos_weight, full_train_loader):
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)

    optimizer = get_optimizer(model, base_lr=best_lr, classifier_lr=1e-4, weight_decay=0.01)

    total_steps = len(full_train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    pos_weight_tensor = torch.tensor([best_pos_weight]).to(device)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    for epoch in range(EPOCHS):
        model.train()
        loop = tqdm(full_train_loader, desc=f"Retrain | Epoch {epoch+1}/{EPOCHS}", leave=False)
        for batch in loop:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].float().to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits.view(-1), labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            loop.set_postfix(loss=loss.item())

    return model

#Training Loop
all_results = []

for label_to_train in custom_labels:
    safe_label = label_to_train.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_bert_model_{safe_label}.pt"

    if model_path.exists():
        print(f" Skipping {label_to_train} — model already exists.")
        continue

    print(f"\nTraining for: {label_to_train}")

    train_df = sam[sam['policy_type'] == 'TRAIN'].copy()
    train_df['binary_label'] = train_df['value_name'].apply(lambda x: 1 if label_to_train in x else 0)

    positives = train_df[train_df['binary_label'] == 1]
    negatives = train_df[train_df['binary_label'] == 0]
    if len(positives) < 5:
        print(f" Too few positives ({len(positives)}). Skipping.")
        continue

    # Compute pos_weight on original training data
    label_counts = Counter(train_df['binary_label'].tolist())
    pos_weight = label_counts[0] / label_counts[1]
    print(f" Using pos_weight: {pos_weight:.2f}")

    # rain/val split
    texts = train_df['segment_text'].tolist()
    labels = train_df['binary_label'].tolist()
    texts_train, texts_val, y_train, y_val = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=SEED
    )
    with open(val_save_path, "wb") as f:
        pickle.dump({"texts": texts_val, "labels": y_val}, f)

    # Oversample positives in training
    train_data = pd.DataFrame({'segment_text': texts_train, 'binary_label': y_train})
    positives_train = train_data[train_data['binary_label'] == 1]
    negatives_train = train_data[train_data['binary_label'] == 0]

    max_oversample_size = min(len(negatives_train), len(positives_train) * 4)
    oversampled_positives = positives_train.sample(max_oversample_size, replace=True, random_state=SEED)
    train_df_balanced = pd.concat([negatives_train, oversampled_positives], ignore_index=True).sample(frac=1, random_state=SEED)

    # Build loaders
    train_dataset = TextDataset(train_df_balanced['segment_text'].tolist(), train_df_balanced['binary_label'].tolist())
    val_dataset = TextDataset(texts_val, y_val)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # Find best model
    best_f1 = 0
    best_lr = None
    for lr in LEARNING_RATES:
        model = train_model(lr, train_loader, pos_weight)
        acc, f1, precision, recall = evaluate_model(model, val_loader)

        if f1 > best_f1:
            best_f1 = f1
            best_lr = lr
            torch.save(model.state_dict(), model_path)
            print(f"Best model updated and saved at LR {lr} for label {label_to_train}")

        all_results.append({
            'label': label_to_train,
            'learning_rate': lr,
            'pos_weight': round(pos_weight, 2),
            'f1': round(f1, 4),
            'precision': round(precision, 4),
            'recall': round(recall, 4),
            'best_model': 'best' if f1 == best_f1 else ''
        })

        # Save results to excel immediately after every LR
        all_models_df = pd.DataFrame(all_results)

        # Append or create "All Models" sheet
        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "All Models" in book.sheetnames:
                existing_df = pd.read_excel(excel_path, sheet_name="All Models")
                combined_df = pd.concat([existing_df, all_models_df], ignore_index=True)
                combined_df.drop_duplicates(subset=['label', 'learning_rate'], inplace=True)
            else:
                combined_df = all_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_df.to_excel(writer, sheet_name="All Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                all_models_df.to_excel(writer, sheet_name="All Models", index=False)

        # Append or create "Best Models" sheet
        best_models_df = all_models_df.sort_values('f1', ascending=False).drop_duplicates('label')

        if os.path.exists(excel_path):
            book = load_workbook(excel_path)
            if "Best Models" in book.sheetnames:
                existing_best_df = pd.read_excel(excel_path, sheet_name="Best Models")
                combined_best_df = pd.concat([existing_best_df, best_models_df], ignore_index=True)
                combined_best_df.drop_duplicates(subset=['label'], inplace=True)
            else:
                combined_best_df = best_models_df

            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                combined_best_df.to_excel(writer, sheet_name="Best Models", index=False)
        else:
            with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
                best_models_df.to_excel(writer, sheet_name="Best Models", index=False)



    # Final retrain on full dataset
    full_dataset = TextDataset(train_df['segment_text'].tolist(), train_df['binary_label'].tolist())
    full_loader = DataLoader(full_dataset, batch_size=BATCH_SIZE, shuffle=True)

    print(f"\n Retraining best model for {label_to_train} on full data...")
    best_model = retrain_best_model(best_lr, pos_weight, full_loader)

    torch.save(best_model.state_dict(), model_path)
    print(f" Final model saved to {model_path}")


## Test

In [ ]:

# Config
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/win")
TEST_EXCEL_PATH = "/content/drive/MyDrive/Colab Notebooks/Project/win/test_results_bert.xlsx"
BATCH_SIZE = 32
MAX_LEN = 512

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

custom_labels = [
    'Information Type_Contact information',
    'Purpose_Essential service or feature',
    'Information Type_Personal identifier',
    "Information Type_IP address and device IDs",
    "Information Type_Location",
    "Information Type_Health, genetic, or biometric data",
    "Information Type_Computer information",
    "Information Type_User online activities",
    "Information Type_Generic personal information",
    "Collection Process_Collected on first-party website/app",
    "Collection Process_Shared by first party with a third party",
    "Purpose_Advertising or marketing",
    "Purpose_Analytics or research",
    "Purpose_Service operation and security",
    "Purpose_Legal requirement"
]


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

# Evaluation
def evaluate_model(model, dataloader):
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().tolist())

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs.logits).squeeze(-1)
            probs = probs.detach().cpu().numpy().flatten()
            preds.extend(probs)

    bin_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(true_labels, bin_preds)
    f1 = f1_score(true_labels, bin_preds, zero_division=0)
    precision = precision_score(true_labels, bin_preds, zero_division=0)
    recall = recall_score(true_labels, bin_preds, zero_division=0)

    return acc, f1, precision, recall

# Load models, test and save results
results = []

for label_to_test in tqdm(custom_labels, desc="Testing all labels"):
    safe_label = label_to_test.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_bert_model_{safe_label}.pt"

    if not model_path.exists():
        print(f" Model file missing for {label_to_test}, skipping.")
        continue

    print(f"\n Evaluating model for label: {label_to_test}")

    # Prepare test data
    test_df = sam[sam['policy_type'] == 'TEST'].copy()
    test_df['binary_label'] = test_df['value_name'].apply(lambda x: 1 if label_to_test in x else 0)

    test_dataset = TextDataset(test_df['segment_text'].tolist(), test_df['binary_label'].tolist())
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    # Load model
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))

    acc, f1, precision, recall = evaluate_model(model, test_loader)
    print(f" Results: Acc={acc:.4f}, F1={f1:.4f}, Precision={precision:.4f}, Recall={recall:.4f}\n")

    results.append({
        'label': label_to_test,
        'accuracy': round(acc, 4),
        'f1_score': round(f1, 4),
        'precision': round(precision, 4),
        'recall': round(recall, 4),
    })

# Save results
results_df = pd.DataFrame(results)
results_df.to_excel(TEST_EXCEL_PATH, index=False)
print(f"\n Test results saved to: {TEST_EXCEL_PATH}")


## Threshold Tunning

In [ ]:
# Constants
MAX_LEN = 512
BATCH_SIZE = 32
SEED = 42
MODEL_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project/win")
excel_path_training = MODEL_DIR / "training_results.xlsx"
excel_path_test = MODEL_DIR / "test_results.xlsx"

# Initialize tokenizer and device
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TextDataset(Dataset):
    def __init__(self, texts, labels):
        encodings = tokenizer(texts, truncation=True, padding='max_length', max_length=MAX_LEN)
        self.input_ids = torch.tensor(encodings['input_ids'])
        self.attn_mask = torch.tensor(encodings['attention_mask'])
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attn_mask[idx],
            'labels': self.labels[idx]
        }

# Prediction
def get_raw_predictions(model, dataloader):
    model.eval()
    probs_list, true_labels = [], []
    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            true_labels.extend(labels.cpu().numpy())
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits.squeeze()
            probs = torch.sigmoid(logits)
            probs_list.extend(probs.cpu().numpy())
    return np.array(true_labels), np.array(probs_list)

def find_best_threshold(y_true, y_probs):
    best_thr, best_f1 = 0.5, 0
    for thr in np.linspace(0, 1, 101):
        preds = (y_probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr, best_f1

# Load the "Best Models" sheet
test_results = pd.read_excel(excel_path_test)
results_comparison = []

# Loop for all labels
for label in test_results['label']:
    safe_label = label.replace(" ", "_").replace("/", "_")
    model_path = MODEL_DIR / f"best_bert_model_{safe_label}.pt"
    val_save_path = MODEL_DIR / f"val_set_{safe_label}.pkl"

    if not model_path.exists():
        print(f" No saved model for {label}, skipping.")
        continue

    print(f"\n Processing label: {label}")
    model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=1).to(device)
    model.load_state_dict(torch.load(model_path))

    #Load or create validation split
    if val_save_path.exists():
        with open(val_save_path, "rb") as f:
            val_data = pickle.load(f)
        texts_val, y_val = val_data["texts"], val_data["labels"]
    else:
        df_train = sam[sam['policy_type'] == 'TRAIN'].copy()
        df_train['binary_label'] = df_train['value_name'].apply(lambda x: 1 if label in x else 0)
        _, texts_val, _, y_val = train_test_split(
            df_train['segment_text'].tolist(),
            df_train['binary_label'].tolist(),
            test_size=0.2,
            stratify=df_train['binary_label'],
            random_state=SEED
        )
        with open(val_save_path, "wb") as f:
            pickle.dump({"texts": texts_val, "labels": y_val}, f)

    val_dataset = TextDataset(texts_val, y_val)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    # === Find best threshold ===
    y_val_true, y_val_probs = get_raw_predictions(model, val_loader)
    best_thr, best_val_f1 = find_best_threshold(y_val_true, y_val_probs)
    print(f" Best threshold for {label} = {best_thr:.2f} (Val F1={best_val_f1:.4f})")

    # === Test data ===
    test_df = sam[sam['policy_type'] == 'TEST'].copy()
    test_df['binary_label'] = test_df['value_name'].apply(lambda x: 1 if label in x else 0)
    test_dataset = TextDataset(test_df['segment_text'].tolist(), test_df['binary_label'].tolist())
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    y_test_true, y_test_probs = get_raw_predictions(model, test_loader)

    # Before & After thresholds
    preds_before = (y_test_probs >= 0.5).astype(int)
    preds_after = (y_test_probs >= best_thr).astype(int)

    acc_before, f1_before = accuracy_score(y_test_true, preds_before), f1_score(y_test_true, preds_before, zero_division=0)
    prec_before, rec_before = precision_score(y_test_true, preds_before, zero_division=0), recall_score(y_test_true, preds_before, zero_division=0)
    acc_after, f1_after = accuracy_score(y_test_true, preds_after), f1_score(y_test_true, preds_after, zero_division=0)
    prec_after, rec_after = precision_score(y_test_true, preds_after, zero_division=0), recall_score(y_test_true, preds_after, zero_division=0)

    print(f" Test BEFORE:  F1={f1_before:.4f} Prec={prec_before:.4f} Rec={rec_before:.4f}")
    print(f" Test AFTER:   F1={f1_after:.4f} Prec={prec_after:.4f} Rec={rec_after:.4f}")

    results_comparison.append({
        "Label": label,
        "Best Threshold": best_thr,
        "Before F1": f1_before,
        "After F1": f1_after,
        "Before Precision": prec_before,
        "After Precision": prec_after,
        "Before Recall": rec_before,
        "After Recall": rec_after,
        "Before Accuracy": acc_before,
        "After Accuracy": acc_after,
        "F1 Improvement": f1_after - f1_before
    })

# === Save results ===
comparison_df = pd.DataFrame(results_comparison)
try:
    book = load_workbook(excel_path_test)

    with pd.ExcelWriter(excel_path_test, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        comparison_df.to_excel(writer, sheet_name='Threshold Comparison', index=False)

    print(f"\n Appended results to '{excel_path_test}' in sheet 'Threshold Comparison'.")
except FileNotFoundError:
    # If file doesn't exist, create it with the sheet
    comparison_df.to_excel(excel_path_test, sheet_name='Threshold Comparison', index=False)
    print(f"\n Created '{excel_path_test}' and saved results in sheet 'Threshold Comparison'.")

## Change to Torchscript

In [ ]:
import shutil
from pathlib import Path
import torch
from transformers import BertForSequenceClassification
import glob

input_model_dir = Path("/content/drive/MyDrive/Colab Notebooks/Project/win")
output_model_dir = Path("/content/drive/MyDrive/Colab Notebooks/Project/torch")
output_model_dir.mkdir(exist_ok=True, parents=True)

LABELS = [
    "Information Type_IP address and device IDs",
    "Information Type_Contact information",
    "Information Type_Location",
    "Information Type_Personal identifier",
    "Information Type_Health, genetic, or biometric data",
    "Information Type_Computer information",
    "Information Type_User online activities",
    "Information Type_Generic personal information",
    "Collection Process_Collected on first-party website/app",
    "Collection Process_Shared by first party with a third party",
    "Purpose_Advertising or marketing",
    "Purpose_Analytics or research",
    "Purpose_Essential service or feature",
    "Purpose_Service operation and security",
    "Purpose_Legal requirement"
]

def safe_label(label):
    return label.replace(" ", "_").replace("/", "_")

class TorchScriptWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits

for label in LABELS:
    safe = safe_label(label)
    pattern = str(input_model_dir / f"best_bert_model_{safe}*.pt")
    matching_files = glob.glob(pattern)

    if not matching_files:
        print(f"No model file found for pattern: {pattern}, skipping.")
        continue

    input_path = Path(matching_files[0])
    output_path = output_model_dir / f"torchscript_{safe}.pt"

    print(f"Loading model: {input_path}")
    base_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=1)
    base_model.load_state_dict(torch.load(input_path, map_location="cpu"))
    base_model.eval()

    wrapper = TorchScriptWrapper(base_model)
    dummy_input_ids = torch.randint(0, 30522, (1, 512))
    dummy_attention_mask = torch.ones((1, 512), dtype=torch.long)

    scripted_model = torch.jit.trace(wrapper, (dummy_input_ids, dummy_attention_mask))
    torch.jit.save(scripted_model, str(output_path))

    print(f"Saved TorchScript model to {output_path}")
